In this code, the Land Surface Temperature (LST) is computed and through the use of zonal statistics the LST is computed at the ward level for Bangalore.

There are many papers that follow this methodology. As an example, we follow the Javascript (JS) GEE script from quite some resources on the internet. One such resource is this [Github resource](https://github.com/waleedgeo/EarthEngineProjects/blob/main/projects/p3-lst-uhi/main.JS) and ([YouTube video](https://www.youtube.com/watch?v=5W84zme9QmE)). However, the code was adapted based on the some observations for Bangalore and things in the literature.

In order to translate the JS script into Python code, we use the Geemap "[geemap.js_snippet_to_py](https://book.geemap.org/chapters/03_gee_data.html#converting-javascript-to-python)" command to do the conversion. Of course, we double check the syntax and ran the script also in JS for Bangalore.

In [ ]:
from pathlib import Path

import ee
import geemap
import geopandas as gpd
import pandas as pd

In [ ]:
cloud_project = 'FILL IN YOUR CLOUD PROJECT'

try:
    ee.Initialize(project=cloud_project)
except:
    ee.Authenticate()
    ee.Initialize(project=cloud_project)

In [ ]:
# Define the paths
BASE_DIR = Path.cwd().parent
print(BASE_DIR)

DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

In [ ]:
## Detailed wards
df_inpath = PROCESSED_DIR / "chennai_fix_mapshaper.shp"
print(df_inpath)

In [ ]:
# https://geemap.org/notebooks/10_shapefiles/
chennai = geemap.shp_to_ee(df_inpath)

In [ ]:
print(f'The number of wards in Chennai is {chennai.size().getInfo()}')

In [ ]:
chennaiGeometry = chennai.union(50)

Map = geemap.Map()
Map.centerObject(chennaiGeometry, zoom=10)
Map

In [ ]:
Map.addLayer(chennaiGeometry, {}, 'Chennai outline')
Map

In [ ]:
# Add the ward data
image = ee.Image().paint(chennai, 0, 2)
Map.addLayer(image, {'palette': 'red'}, "Chennai geometry")
Map

## Computing the Land Surface Temperature

In [ ]:
# https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2#colab-python
# Applies scaling factors.
def apply_scale_factors(image):
  optical_bands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
  thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
  return image.addBands(optical_bands, None, True).addBands(
      thermal_bands, None, True
  )

In [ ]:
# Cloud mask obtained from JS script:
# https://github.com/waleedgeo/EarthEngineProjects/blob/main/projects/p3-lst-uhi/main.JS
def cloudmask(image):
    # Script obtained "from":
    # https":#github.com/waleedgeo/EarthEngineProjects/blob/main/projects/p3-lst-uhi/main.JS
    # Bits 3 and 5 are cloud shadow and cloud, respectively.
    # Changed to be applied to image
    cloudShadowBitMask = (1 << 3)
    cloudsBitMask = (1 << 5)
    # Get the pixel QA band.
    qa = image.select('QA_PIXEL')
    # Both flags should be set to zero
    mask = qa.bitwiseAnd(cloudShadowBitMask).eq(0) \
    .And(qa.bitwiseAnd(cloudsBitMask).eq(0))
    return image.updateMask(mask)

In [ ]:
colThsummer = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
.filterBounds(chennaiGeometry) \
.filterDate('2024-01-01', '2025-11-01') \
.filter(ee.Filter.calendarRange(3, 5, 'month')) \
.map(apply_scale_factors) \
.map(cloudmask)

## print('Landsat collection for thermal values', colThsummer.getInfo())

In [ ]:
## colThsummer

In [ ]:
imageThsummer = colThsummer.median()

In [ ]:
image_clipped = imageThsummer.clip(chennaiGeometry)
# https://tutorials.geemap.org/Image/image_visualization/
visualisation = {
  'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 
  'min': 0,
  'max': 0.5,
  'gamma': [0.95, 1.1, 1]
}

Map.addLayer(image_clipped, visualisation, 'True Color Composite Chennai')

Map

A water mask is created as this may distort the LST computations (Chakraborty (2023)). This is mostly not done but since we are only interested in the LST on the ground, we can leave out the water pixels.

In [ ]:
ndvi = imageThsummer \
.normalizedDifference(['SR_B5', 'SR_B4']) \
.rename('NDVI') \
.clip(chennaiGeometry) 

In [ ]:
Map.addLayer(ndvi, {
    "min": 0,
    "max": 1,
    "palette": ['blue', 'white', 'green']
},
'ndvi')
Map

In [ ]:
## Code translated from JS code: 
## https://tinyurl.com/34pmp8jh
## EEFAbook - Chakraborty (2023)
minMax = ndvi.reduceRegion(
reducer=ee.Reducer.min().combine(
reducer2=ee.Reducer.max(),
sharedInputs=True
),
geometry=chennaiGeometry,
scale=30,
maxPixels=1e9
)
print('minMax', minMax.getInfo())

In [ ]:
min = ee.Number(minMax.get('NDVI_min'))
max = ee.Number(minMax.get('NDVI_max'))
min

In [ ]:
max

Although this is min-max is widely used, this can inflate the fractional cover due to the minimum values being negative. You can work out an example using the following values: assume NDVI minimum is - 0.3, NDVI max is 0.7 and 0.4 is the NDVI value of an urban vegetation pixel. You get for the fractional vegetation (please see below) a value of 0.7 which is an overestimation. An alternative would be to use percentiles but even then, the NDVI max is low:

In [ ]:
## Alternative - work with percentiles
# ndviPerc = ndvi.reduceRegion(
# reducer =ee.Reducer.percentile([5, 95]),
# geometry = bangGeometry,
# scale = 30,
# maxPixels = 1e13
# )

# ndviMin = ee.Number(ndviPerc.get('NDVI_p5'))
# ndviMax = ee.Number(ndviPerc.get('NDVI_p95'))

# print("NDVIp5:", ndviMin.getInfo())
# print("NDVIp95:", ndviMax.getInfo())

As an alternative, we will use fixed values. This issue is also pointed out in a recent paper of Bobáľováa and Opravilb (2025). Widely used fixed numbers are 0.2 for the minimum ad 0.5 for the maximum NDVI values.

The formula for the fractional vegetation is given below. Please note that this is squared as per the nonlinear scaling proposed by Carlson and Ripley (1997) and adopted by Sobrino et al. (2004), where the squared NDVI fraction better represents vegetation area in mixed pixels. This is widely adopted in the literature of Land Surface Temperature. Moreover, the values are clamped such that they fall within the [0,1] range.

In [ ]:
min = ee.Number(0.2) # Values of Ermida et al. (2020)
max = ee.Number(0.86) 

In [ ]:
fv = ndvi \
.subtract(min) \
.divide(max.subtract(min)) \
.pow(2) \
.clamp(0, 1) \
.rename('FV');

In [ ]:
## fv

In [ ]:
Map.addLayer(fv, {
    "min": 0,
    "max": 1,
    "palette": ['blue', 'white', 'green']}, 'fv')
Map

In [ ]:
## Calculate emissivity
a = ee.Number(0.004)
b = ee.Number(0.986)
em = fv.multiply(a).add(b).rename('EMM')

Map.addLayer(em, {
    "min": 0.98,
    "max": 0.99,
    "palette": ['blue', 'white', 'green']},'EMM')

Map

In [ ]:
thermalsummer = imageThsummer.select('ST_B10') \
.clip(chennaiGeometry) \

Map.addLayer(thermalsummer, {
"min": 280,
"max": 330,
"palette": ['blue', 'white', 'red']},'Landsat_BT')

Map

In [ ]:
lstLandsatsummer = thermalsummer.expression(
'(Tb/(1 + (0.001145* (Tb / 1.438))*log(Ep)))-273.15', {
    'Tb': thermalsummer.select('ST_B10'),
    'Ep': em.select('EMM')
})

In [ ]:
Map.addLayer(lstLandsatsummer, {
    "min": 35,
    "max": 42,
    "palette": ['blue', 'white', 'red'],
},
'LST_Landsat')

vis_params = {'min': 35, 'max': 42, 'palette': ['blue', 'white', 'red']}
Map.add_colorbar(vis_params, label='Land Surface Temperature (°C)',position= 'bottomleft')
Map

## Zonal statistics for LST

In [ ]:
lstwardsmanualsummer = lstLandsatsummer.reduceRegions(
collection=chennai,
reducer=ee.Reducer.mean(),
scale=30,
)

In [ ]:
## lstwardsmanualsummer

In [ ]:
lstwardsmanualsummer.select(['Name', 'zone_id', 'zone_name', 'mean'])

In [ ]:
outdir = PROCESSED_DIR /'lstcalcsummer_chennai.csv'
print(outdir)

In [ ]:
geemap.ee_export_vector(lstwardsmanualsummer, outdir, verbose=True)

In [ ]:
df_lst_mean_summer = pd.read_csv(outdir)

In [ ]:
df_lst_mean_summer

In [ ]:
print(f"The minimum temperature is {df_lst_mean_summer['mean'].min()}.")
print(f"The maximum temperature is {df_lst_mean_summer['mean'].max()}.")

In [ ]:
df_wards = gpd.read_file(df_inpath)
df_wards.head()

In [ ]:
df_wards_merged_summer = df_wards.merge(df_lst_mean_summer, on = 'Name')

In [ ]:
df_wards_merged_summer

In [ ]:
df_wards_merged_summer = df_wards_merged_summer[['Name','zone_id_x', 'zone_name_x', 'geometry', 'mean']]

In [ ]:
df_wards_merged_summer = df_wards_merged_summer.rename(columns={
    "zone_id_x": "zone_id",
    "zone_name_x": "zone_name"
})

In [ ]:
outdirlst = PROCESSED_DIR / 'merged_lst_wards_summer_calc_chennai.shp'
print(outdirlst)

In [ ]:
df_wards_merged_summer.to_file(outdirlst)

In [ ]:
df_wards_merged_summer.shape

In [ ]:
fig_path =  BASE_DIR / 'images' /'lst_ward_summer_chennai.png'

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

# IMPORTANT: capture fig and ax
fig, ax = plt.subplots(1, 1, figsize=(14, 10))

divider = make_axes_locatable(ax)
cax = divider.append_axes("bottom", size="5%", pad=0.1)

df_wards_merged_summer.plot(
    column="mean",
    ax=ax,
    legend=True,
    cax=cax,
    legend_kwds={
        "label": "Mean Land Surface Temperature per ward (°C)",
        "orientation": "horizontal"
    },
)
plt.savefig(fig_path)
ax.set_axis_off()
plt.savefig(fig_path, dpi = 300, bbox_inches = "tight")
plt.show()


## References

T. Chakraborty (2023), "Heat islands,” in Cloud-based remote sensing with Google Earth Engine, J. A. Cardille, M. A. Crowley, D. Saah and N. E.
Clinton, Eds. Springer Open, 2023 [Online] Available: https://link.springer.com/book/10.1007/978-3-031-26588-4

Ermida, S.L., Soares, P., Mantas, V., Göttsche, F.M., Trigo, I.F. (2020). Google earth engine open-source code for land surface temperature estimation from the landsat series. Remote Sens. 12, 1–21. 576 https://doi.org/10.3390/RS12091471

Hana Bobáľová, Šimon Opravil (2025), Improving Landsat land surface temperature estimation in Google Earth Engine using NDVI-based emissivity, Advances in Space Research, https://doi.org/10.1016/j.asr.2025.11.085. Paper accessible through [Eartharxiv](https://eartharxiv.org/repository/view/10586/#:~:text=longest%20temporal%20continuity.-,Although%20Landsat%20Surface%20Temperature%20(ST)%20is%20already%20available%20as%20a,the%20highest%20on%20bare%20soil).

Mirza Waleed (2023), https://github.com/waleedgeo/EarthEngineProjects/blob/main/projects/p3-lst-uhi/main.JS

[Mirza Waleed (2023), YouTube video on LST, Urban Heat Island Effect, and UTFVI Analysis using Google Earth Engine and Landsat dataset]()

Sobrino, J. A., Jiménez-Muñoz, J. C., & Paolini, L. (2004). Land surface temperature retrieval from LANDSAT TM 5. Remote Sensing of environment, 90(4), 434-440.

Wu, Q. (2023). Earth Engine and Geemap. Geospatial Datascience with Python (1st ed.). Locate Press.  https://doi.org/10.21105/joss.02305 

